In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
df = pd.read_csv("../cleaned_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

Dataset loaded successfully!
Shape: (51290, 23)

First 5 rows:
          order_id  order_date   ship_date       ship_mode    customer_name  \
0     AG-2011-2040  2011-01-01  2011-01-06  Standard Class  Toby Braunhardt   
1    IN-2011-47883  2011-01-01  2011-01-08  Standard Class      Joseph Holt   
2     HU-2011-1220  2011-01-01  2011-01-05    Second Class    Annie Thurman   
3  IT-2011-3647632  2011-01-01  2011-01-05    Second Class     Eugene Moren   
4    IN-2011-47883  2011-01-01  2011-01-08  Standard Class      Joseph Holt   

       segment            state    country  market   region  ...  \
0     Consumer      Constantine    Algeria  Africa   Africa  ...   
1     Consumer  New South Wales  Australia    APAC  Oceania  ...   
2     Consumer         Budapest    Hungary    EMEA     EMEA  ...   
3  Home Office        Stockholm     Sweden      EU    North  ...   
4     Consumer  New South Wales  Australia    APAC  Oceania  ...   

                  product_name  sales quantity discou

In [4]:
# Identify numerical and categorical features

numerical_features = df.select_dtypes(include=["int64", "float64"]).columns
categorical_features = df.select_dtypes(include=["object"]).columns

print("Numerical Features:")
print(list(numerical_features))

print("\nCategorical Features:")
print(list(categorical_features))

print("\nNumber of Numerical Features:", len(numerical_features))
print("Number of Categorical Features:", len(categorical_features))

Numerical Features:
['sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'year', 'shipping_days', 'profit_margin']

Categorical Features:
['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'order_priority']

Number of Numerical Features: 8
Number of Categorical Features: 15


C:\Users\arjun\AppData\Local\Temp\ipykernel_23612\1547133389.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=["object"]).columns


In [5]:
# Create new derived features

df["sales_per_shipping_day"] = df["sales"] / (df["shipping_days"] + 1)

df["profit_per_sales"] = df["profit"] / (df["sales"] + 1)

print("New features created successfully!")

print("\nNew features:")
print(df[[
    "sales",
    "profit",
    "shipping_days",
    "sales_per_shipping_day",
    "profit_per_sales"
]].head())

New features created successfully!

New features:
   sales   profit  shipping_days  sales_per_shipping_day  profit_per_sales
0  408.0  106.140              5                   68.00          0.259511
1  120.0   36.036              7                   15.00          0.297818
2   66.0   29.640              4                   13.20          0.442388
3   45.0  -26.055              4                    9.00         -0.566413
4  114.0   37.770              7                   14.25          0.328435


In [6]:
# One-Hot Encoding categorical features

categorical_features = ["category", "segment", "region"]

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoded_data = encoder.fit_transform(df[categorical_features])

encoded_columns = encoder.get_feature_names_out(categorical_features)

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoded_columns,
    index=df.index
)

print("Categorical features encoded successfully!")

print("\nEncoded columns:")
print(list(encoded_columns))

print("\nEncoded data:")
print(encoded_df.head())

Categorical features encoded successfully!

Encoded columns:
['category_Furniture', 'category_Office Supplies', 'category_Technology', 'segment_Consumer', 'segment_Corporate', 'segment_Home Office', 'region_Africa', 'region_Canada', 'region_Caribbean', 'region_Central', 'region_Central Asia', 'region_EMEA', 'region_East', 'region_North', 'region_North Asia', 'region_Oceania', 'region_South', 'region_Southeast Asia', 'region_West']

Encoded data:
   category_Furniture  category_Office Supplies  category_Technology  \
0                 0.0                       1.0                  0.0   
1                 0.0                       1.0                  0.0   
2                 0.0                       1.0                  0.0   
3                 0.0                       1.0                  0.0   
4                 1.0                       0.0                  0.0   

   segment_Consumer  segment_Corporate  segment_Home Office  region_Africa  \
0               1.0                0.0 

In [7]:
# Combine original numerical features with encoded categorical features

numerical_data = df[numerical_features].copy()

feature_engineered_df = pd.concat(
    [numerical_data, encoded_df],
    axis=1
)

print("Feature engineering completed!")

print("\nOriginal dataset shape:")
print(df.shape)

print("\nFeature-engineered dataset shape:")
print(feature_engineered_df.shape)

print("\nFirst 5 rows:")
print(feature_engineered_df.head())

Feature engineering completed!

Original dataset shape:
(51290, 25)

Feature-engineered dataset shape:
(51290, 27)

First 5 rows:
   sales  quantity  discount   profit  shipping_cost  year  shipping_days  \
0  408.0         2       0.0  106.140          35.46  2011              5   
1  120.0         3       0.1   36.036           9.72  2011              7   
2   66.0         4       0.0   29.640           8.17  2011              4   
3   45.0         3       0.5  -26.055           4.82  2011              4   
4  114.0         5       0.1   37.770           4.70  2011              7   

   profit_margin  category_Furniture  category_Office Supplies  ...  \
0      26.014706                 0.0                       1.0  ...   
1      30.030000                 0.0                       1.0  ...   
2      44.909091                 0.0                       1.0  ...   
3     -57.900000                 0.0                       1.0  ...   
4      33.131579                 1.0                

In [9]:
# Check for infinity values

print("Infinity values before cleaning:")
print(np.isinf(feature_engineered_df).sum().sum())

Infinity values before cleaning:
1


In [10]:
# Find infinity values

print("Infinity values by column:")
print(np.isinf(feature_engineered_df).sum())

print("\nRows containing infinity:")
print(feature_engineered_df[np.isinf(feature_engineered_df).any(axis=1)])

Infinity values by column:
sales                       0
quantity                    0
discount                    0
profit                      0
shipping_cost               0
year                        0
shipping_days               0
profit_margin               1
category_Furniture          0
category_Office Supplies    0
category_Technology         0
segment_Consumer            0
segment_Corporate           0
segment_Home Office         0
region_Africa               0
region_Canada               0
region_Caribbean            0
region_Central              0
region_Central Asia         0
region_EMEA                 0
region_East                 0
region_North                0
region_North Asia           0
region_Oceania              0
region_South                0
region_Southeast Asia       0
region_West                 0
dtype: int64

Rows containing infinity:
       sales  quantity  discount  profit  shipping_cost  year  shipping_days  \
40017    0.0         1       0.8   -1.11   

In [11]:
# Replace infinity values with NaN

feature_engineered_df = feature_engineered_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows containing NaN values

feature_engineered_df = feature_engineered_df.dropna()

print("Infinity values after cleaning:")
print(np.isinf(feature_engineered_df).sum().sum())

print("\nNew shape:")
print(feature_engineered_df.shape)

Infinity values after cleaning:
0

New shape:
(51289, 27)


In [13]:
# Add derived features to feature-engineered dataset

feature_engineered_df["sales_per_shipping_day"] = (
    df.loc[feature_engineered_df.index, "sales_per_shipping_day"]
)

feature_engineered_df["profit_per_sales"] = (
    df.loc[feature_engineered_df.index, "profit_per_sales"]
)

print("Derived features added successfully!")

print("\nFeature-engineered dataset shape:")
print(feature_engineered_df.shape)

print("\nNew features:")
print(feature_engineered_df[
    ["sales_per_shipping_day", "profit_per_sales"]
].head())

Derived features added successfully!

Feature-engineered dataset shape:
(51289, 29)

New features:
   sales_per_shipping_day  profit_per_sales
0                   68.00          0.259511
1                   15.00          0.297818
2                   13.20          0.442388
3                    9.00         -0.566413
4                   14.25          0.328435


In [14]:
# Scale numerical features

features_to_scale = list(numerical_features) + [
    "sales_per_shipping_day",
    "profit_per_sales"
]

scaler = StandardScaler()

scaled_data = scaler.fit_transform(
    feature_engineered_df[features_to_scale]
)

scaled_df = pd.DataFrame(
    scaled_data,
    columns=features_to_scale,
    index=feature_engineered_df.index
)

print("Numerical features scaled successfully!")

print("\nScaled data:")
print(scaled_df.head())

Numerical features scaled successfully!

Scaled data:
      sales  quantity  discount    profit  shipping_cost      year  \
0  0.331231 -0.647987 -0.673206  0.444306       0.158536 -1.617213   
1 -0.259459 -0.209148 -0.202086  0.042389      -0.290705 -1.617213   
2 -0.370213  0.229691 -0.673206  0.005720      -0.317757 -1.617213   
3 -0.413284 -0.209148  1.682395 -0.313588      -0.376225 -1.617213   
4 -0.271765  0.668530 -0.202086  0.052330      -0.378319 -1.617213   

   shipping_days  profit_margin  sales_per_shipping_day  profit_per_sales  
0       0.595934       0.456122                0.030541          0.473625  
1       1.752380       0.542131               -0.300409          0.560493  
2       0.017711       0.860846               -0.311649          0.888330  
3       0.017711      -1.341358               -0.337875         -1.399300  
4       1.752380       0.608568               -0.305092          0.629921  


In [15]:
# Combine scaled numerical features and encoded categorical features

final_feature_df = pd.concat(
    [scaled_df, encoded_df.loc[scaled_df.index]],
    axis=1
)

print("Final feature-engineered dataset created!")

print("\nFinal shape:")
print(final_feature_df.shape)

print("\nAre all features numeric?")
print(final_feature_df.dtypes.eq("float64").all())

print("\nFirst 5 rows:")
print(final_feature_df.head())

Final feature-engineered dataset created!

Final shape:
(51289, 29)

Are all features numeric?
True

First 5 rows:
      sales  quantity  discount    profit  shipping_cost      year  \
0  0.331231 -0.647987 -0.673206  0.444306       0.158536 -1.617213   
1 -0.259459 -0.209148 -0.202086  0.042389      -0.290705 -1.617213   
2 -0.370213  0.229691 -0.673206  0.005720      -0.317757 -1.617213   
3 -0.413284 -0.209148  1.682395 -0.313588      -0.376225 -1.617213   
4 -0.271765  0.668530 -0.202086  0.052330      -0.378319 -1.617213   

   shipping_days  profit_margin  sales_per_shipping_day  profit_per_sales  \
0       0.595934       0.456122                0.030541          0.473625   
1       1.752380       0.542131               -0.300409          0.560493   
2       0.017711       0.860846               -0.311649          0.888330   
3       0.017711      -1.341358               -0.337875         -1.399300   
4       1.752380       0.608568               -0.305092          0.629921   

 

In [16]:
# Compare model readiness before vs after feature engineering

print("BEFORE FEATURE ENGINEERING")
print("--------------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Categorical columns:", len(df.select_dtypes(include=["object"]).columns))
print("Numerical columns:", len(df.select_dtypes(include=["number"]).columns))


print("\nAFTER FEATURE ENGINEERING")
print("-------------------------")
print("Rows:", final_feature_df.shape[0])
print("Columns:", final_feature_df.shape[1])
print("All features numeric:", final_feature_df.dtypes.eq("float64").all())
print("Missing values:", final_feature_df.isnull().sum().sum())

BEFORE FEATURE ENGINEERING
--------------------------
Rows: 51290
Columns: 25
Categorical columns: 15
Numerical columns: 10

AFTER FEATURE ENGINEERING
-------------------------
Rows: 51289
Columns: 29
All features numeric: True
Missing values: 0


C:\Users\arjun\AppData\Local\Temp\ipykernel_23612\4129359819.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("Categorical columns:", len(df.select_dtypes(include=["object"]).columns))


In [17]:
final_feature_df.to_csv("feature_engineered_dataset.csv", index=False)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!


In [18]:
import os

print("File exists:", os.path.exists("feature_engineered_dataset.csv"))

print("Final dataset shape:", final_feature_df.shape)

print("\nFinal columns:")
print(final_feature_df.columns.tolist())

File exists: True
Final dataset shape: (51289, 29)

Final columns:
['sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'year', 'shipping_days', 'profit_margin', 'sales_per_shipping_day', 'profit_per_sales', 'category_Furniture', 'category_Office Supplies', 'category_Technology', 'segment_Consumer', 'segment_Corporate', 'segment_Home Office', 'region_Africa', 'region_Canada', 'region_Caribbean', 'region_Central', 'region_Central Asia', 'region_EMEA', 'region_East', 'region_North', 'region_North Asia', 'region_Oceania', 'region_South', 'region_Southeast Asia', 'region_West']
